### Tools


Models can request to call  tools  that  perform  tasks  such  as fetching data from a database,

seraching the web, or running code .Tools are pairings of:

1. A schema,including the name of the tool, a description, and/or argument defination,

   and /or argument definations (often a JSON schema)

2. A function or coroutine to execute.

In [1]:

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3
)


In [2]:
response=model.invoke("why do parrots talk?")
response

AIMessage(content='Parrots “talk” because they’re natural mimics—and because they’re social, intelligent birds that use sound to interact with their environment. Here’s the low‑down:\n\n| Why they mimic | How it works | What it means for you |\n|----------------|--------------|-----------------------|\n| **Vocal learning** | Parrots have a highly developed *syrinx* (their “voice box”) that can produce a huge range of sounds. Their brains are wired to learn and reproduce those sounds, much like human speech learning. | You’ll hear a parrot repeat words, phrases, or even entire songs it hears. |\n| **Social bonding** | In the wild, parrots use vocalizations to stay in contact with flock mates, signal danger, or coordinate foraging. Mimicking a human’s voice can be a way to join the “conversation.” | Your parrot may “talk” to feel part of the family or to get your attention. |\n| **Reward‑driven behavior** | When a parrot says a word and gets a treat, praise, or a fun interaction, it lear

In [3]:
from langchain_core.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather  at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("what's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call the get_weather function.', 'tool_calls': [{'id': 'fc_8e4a15c6-0013-4956-808b-3cf0fc2d1548', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 129, 'total_tokens': 162, 'completion_time': 0.039006218, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.007546398, 'prompt_tokens_details': None, 'queue_time': 0.289921931, 'total_time': 0.046552616}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1e841-e2e2-7278-aa80-03509b90de62-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_8e4a15c6-0013-4956-808b-3cf0fc2d1548', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 129, 'output_t

### Tool Execution Loops

In [5]:
#Step 1: Model generates tool calls
messages= [{"role":"user","content":"what's the weather in boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
#step 2 :Execute tools  and collect  results
for tool_call in ai_msg.tool_calls:
    #execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
#Step 3 : Pass results back to model  for final response 
final_response = model_with_tools.invoke(messages)
print(final_response.text)
#"The  current  weather in boston is 72 F and sunny."

It’s sunny in Boston.


In [6]:
messages

[{'role': 'user', 'content': "what's the weather in boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_248f442b-fba5-4797-8bbe-75531dcac807', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 129, 'total_tokens': 161, 'completion_time': 0.035610755, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006174513, 'prompt_tokens_details': None, 'queue_time': 0.285687286, 'total_time': 0.041785268}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1e86c-12be-705c-9792-87d8371bc374-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_248f442b-fba5-4797-8bbe-75531dcac807', 'type': 'tool_cal

## Remember this simple distinction:
- First invoke = LLM decides what needs to be done.
- Tool invoke = Python actually does the work.
- Second LLM invoke = LLM understands the result and answers.